# B05 Type 1 LLM Pipeline Evaluation

Notebook này dùng để đánh giá nhanh pipeline Type 1 trên Kaggle/local bằng một phần nhỏ của tập dữ liệu có nhãn.

Mục tiêu:
- Chỉ chạy Logic Type 1.
- Mặc định ưu tiên file có gold labels `Logic_Based_Educational_Queries.json`, không dùng inference file nếu bạn muốn đo accuracy.
- Chọn subset nhỏ bằng stratified sampling theo `question_type` và `gold_answer` để thống kê đỡ lệch hơn lấy các dòng đầu.
- Chạy LLM qua codebase hiện tại: `exact.llm_client.build_json_client_from_settings`.
- Ưu tiên vLLM/OpenAI-compatible nếu có `EXACT_LLM_BASE_URL`; nếu không có thì fallback Transformers local.
- Gọi pipeline thật `exact.logic.pipeline.run_type1_pipeline`, không inline solver/parser trong notebook.
- LLM-only: Type 1 không có heuristic fallback.
- Ghi prediction, report JSON, bảng wrong/error cases và thống kê accuracy/error/fallback.

Các biến Kaggle hay dùng:
- `EXACT_LIMIT=24`: số câu cần chạy thử.
- `EXACT_FAST_LOCAL=1`: mặc định khi không có vLLM; giảm LLM calls để tránh quá 60s/request.
- `EXACT_LLM_MODEL=Qwen/Qwen2.5-1.5B-Instruct`: mặc định cho local fast smoke eval; dùng 7B nên chạy qua vLLM/AWQ hoặc endpoint remote.
- `EXACT_REQUEST_TIMEOUT_SECONDS=55`: timeout từng request trong notebook.
- `EXACT_SAMPLE_MODE=stratified`: `stratified`, `random`, hoặc `head`.
- `EXACT_LLM_BASE_URL=http://127.0.0.1:8000/v1`: endpoint vLLM/OpenAI-compatible.
- `EXACT_LLM_MODEL=Qwen/Qwen2.5-7B-Instruct`: tên model đúng với vLLM.
- `EXACT_START_VLLM=1`: tự start vLLM từ notebook nếu muốn.
- `EXACT_INSTALL_DEPS=1`: tự cài dependency Python còn thiếu trên Kaggle.

In [ ]:
from __future__ import annotations

import importlib.util
import json
import os
import random
import subprocess
import sys
import time
import traceback
from collections import Counter, defaultdict
from pathlib import Path



def find_project_root() -> Path:
    candidates = [Path.cwd(), *Path.cwd().parents]
    candidates.extend([
        Path('/kaggle/working/Exact2026'),
        Path('/kaggle/working'),
    ])
    input_root = Path('/kaggle/input')
    if input_root.exists():
        candidates.extend(path for path in input_root.rglob('*') if path.is_dir() and path.name in {'Exact2026', 'exact2026'})
        candidates.extend(path.parent.parent for path in input_root.rglob('src/exact') if path.is_dir())

    seen: set[Path] = set()
    for candidate in candidates:
        candidate = candidate.resolve()
        if candidate in seen:
            continue
        seen.add(candidate)
        if (candidate / 'src' / 'exact').exists():
            return candidate
    raise FileNotFoundError('Cannot find project root containing src/exact. Add the repo as a Kaggle Dataset or copy it to /kaggle/working/Exact2026.')


PROJECT_ROOT = find_project_root()
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print('PROJECT_ROOT =', PROJECT_ROOT)
print('SRC_DIR      =', SRC_DIR)

# Kaggle images vary. Install only small missing runtime deps by default on Kaggle.
IN_KAGGLE = Path('/kaggle/working').exists()
INSTALL_DEPS = os.getenv('EXACT_INSTALL_DEPS', '1' if IN_KAGGLE else '0') == '1'
REQUIRED_MODULES = {
    'pandas': 'pandas>=2.0',
    'pydantic': 'pydantic>=2.0',
    'pydantic_settings': 'pydantic-settings>=2.10.6',
    'openai': 'openai>=1.40',
}
missing = [package for module, package in REQUIRED_MODULES.items() if importlib.util.find_spec(module) is None]
if missing and INSTALL_DEPS:
    print('Installing missing deps:', missing)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
elif missing:
    print('Missing deps:', missing)
    print('Run this cell with EXACT_INSTALL_DEPS=1 or install them manually before importing exact.*')
else:
    print('Core deps available')
import pandas as pd


In [ ]:
# Evaluation knobs.
# Trên Kaggle: mặc định dùng vLLM + 7B AWQ (không cần set env var).
# Local: mặc định dùng Transformers 1.5B (FAST_LOCAL).
# Override bất kỳ giá trị nào bằng env var tương ứng nếu cần.
LLM_BASE_URL = os.getenv('EXACT_LLM_BASE_URL', '').strip() or None
START_VLLM_REQUESTED = os.getenv('EXACT_START_VLLM', '1' if IN_KAGGLE else '0') == '1'
FAST_LOCAL = os.getenv('EXACT_FAST_LOCAL', '1' if not LLM_BASE_URL and not START_VLLM_REQUESTED else '0') == '1'
DEFAULT_MODEL = (
    'Qwen/Qwen2.5-7B-Instruct-AWQ'
    if LLM_BASE_URL or START_VLLM_REQUESTED
    else 'Qwen/Qwen2.5-1.5B-Instruct'
    if FAST_LOCAL
    else 'Qwen/Qwen2.5-7B-Instruct'
)
MODEL_NAME = os.getenv('EXACT_LLM_MODEL', DEFAULT_MODEL)
MAX_NEW_TOKENS = int(os.getenv('EXACT_MAX_NEW_TOKENS', '1024' if FAST_LOCAL else '2048'))
LLM_TEMPERATURE = float(os.getenv('EXACT_LLM_TEMPERATURE', '0.0'))
LLM_TOP_P = float(os.getenv('EXACT_LLM_TOP_P', '1.0'))
LLM_DEVICE_MAP = os.getenv('EXACT_LLM_DEVICE_MAP', 'auto').strip() or None
LLM_TORCH_DTYPE = os.getenv('EXACT_LLM_TORCH_DTYPE', 'float16')
LLM_LOCAL_FILES_ONLY = os.getenv('EXACT_LLM_LOCAL_FILES_ONLY', '0') == '1'
LLM_TRUST_REMOTE_CODE = os.getenv('EXACT_LLM_TRUST_REMOTE_CODE', '0') == '1'
TYPE1_TRANSLATION_SAMPLES = int(os.getenv('EXACT_TYPE1_TRANSLATION_SAMPLES', '1' if FAST_LOCAL else '3'))
TYPE1_SAMPLING_TEMPERATURE = float(os.getenv('EXACT_TYPE1_SAMPLING_TEMPERATURE', '0.0' if FAST_LOCAL else '0.7'))
TYPE1_ENABLE_COT_FALLBACK = os.getenv('EXACT_TYPE1_ENABLE_COT_FALLBACK', '0' if FAST_LOCAL else '1') == '1'
REQUEST_TIMEOUT_SECONDS = float(os.getenv('EXACT_REQUEST_TIMEOUT_SECONDS', '55'))
USE_THREAD_TIMEOUT = bool(LLM_BASE_URL or START_VLLM_REQUESTED)
LIMIT = int(os.getenv('EXACT_LIMIT', '24'))
SAMPLE_MODE = os.getenv('EXACT_SAMPLE_MODE', 'stratified').strip().lower()
SAMPLE_SEED = int(os.getenv('EXACT_SAMPLE_SEED', '42'))
CASE_SENSITIVE = os.getenv('EXACT_CASE_SENSITIVE', '0') == '1'
PROGRESS_EVERY = int(os.getenv('EXACT_PROGRESS_EVERY', '5'))
LOG_EVERY = int(os.getenv('EXACT_LOG_EVERY', '1'))
LOG_TRACEBACK = os.getenv('EXACT_LOG_TRACEBACK', '0') == '1'
CONTINUE_ON_ERROR = os.getenv('EXACT_CONTINUE_ON_ERROR', '1') != '0'
CLEAR_KB_CACHE = os.getenv('EXACT_CLEAR_KB_CACHE', '1') != '0'

if FAST_LOCAL:
    print('FAST_LOCAL enabled: 1 translation sample, no CoT fallback. Use vLLM/AWQ for 7B under a 60s target.')
if not LLM_BASE_URL and not START_VLLM_REQUESTED:
    for module in ['torch', 'transformers', 'accelerate']:
        print(f'{module}_available =', importlib.util.find_spec(module) is not None)

default_output_dir = Path('/kaggle/working') if Path('/kaggle/working').exists() else PROJECT_ROOT / 'outputs' / 'logic'
OUTPUT_PATH = Path(os.getenv('EXACT_LOGIC_OUTPUT', str(default_output_dir / 'type1_llm_predictions.json')))
REPORT_PATH = Path(os.getenv('EXACT_LOGIC_REPORT', str(default_output_dir / 'type1_llm_eval_report.json')))
ERRORS_PATH = Path(os.getenv('EXACT_LOGIC_ERRORS', str(default_output_dir / 'type1_llm_eval_errors.csv')))

print({
    'in_kaggle': IN_KAGGLE,
    'base_url': LLM_BASE_URL,
    'start_vllm_requested': START_VLLM_REQUESTED,
    'model': MODEL_NAME,
    'fast_local': FAST_LOCAL,
    'request_timeout_seconds': REQUEST_TIMEOUT_SECONDS,
    'thread_timeout_enabled': USE_THREAD_TIMEOUT,
    'max_new_tokens': MAX_NEW_TOKENS,
    'llm_temperature': LLM_TEMPERATURE,
    'llm_top_p': LLM_TOP_P,
    'type1_translation_samples': TYPE1_TRANSLATION_SAMPLES,
    'type1_sampling_temperature': TYPE1_SAMPLING_TEMPERATURE,
    'type1_enable_cot_fallback': TYPE1_ENABLE_COT_FALLBACK,
    'limit': LIMIT,
    'sample_mode': SAMPLE_MODE,
    'sample_seed': SAMPLE_SEED,
    'case_sensitive': CASE_SENSITIVE,
    'progress_every': PROGRESS_EVERY,
    'log_every': LOG_EVERY,
    'continue_on_error': CONTINUE_ON_ERROR,
    'clear_kb_cache': CLEAR_KB_CACHE,
    'output': str(OUTPUT_PATH),
    'report': str(REPORT_PATH),
    'errors': str(ERRORS_PATH),
})

In [ ]:
# vLLM server launcher.
# Trên Kaggle: tự động start vLLM + install nếu cần (không cần set env var).
# Local: mặc định tắt. Set EXACT_START_VLLM=1 để bật thủ công.
import urllib.error
import urllib.request

START_VLLM = os.getenv('EXACT_START_VLLM', '1' if IN_KAGGLE else '0') == '1'
INSTALL_VLLM = os.getenv('EXACT_INSTALL_VLLM', '1' if IN_KAGGLE else '0') == '1'
VLLM_HOST = os.getenv('EXACT_VLLM_HOST', '127.0.0.1')
VLLM_PORT = int(os.getenv('EXACT_VLLM_PORT', '8000'))
VLLM_MODEL = os.getenv('EXACT_VLLM_MODEL', MODEL_NAME if MODEL_NAME else 'Qwen/Qwen2.5-7B-Instruct-AWQ')
VLLM_SERVED_MODEL_NAME = os.getenv('EXACT_VLLM_SERVED_MODEL_NAME', VLLM_MODEL)
VLLM_QUANTIZATION = os.getenv('EXACT_VLLM_QUANTIZATION', 'awq_marlin' if 'AWQ' in VLLM_MODEL.upper() else '').strip()
VLLM_DTYPE = os.getenv('EXACT_VLLM_DTYPE', 'float16')
VLLM_MAX_MODEL_LEN = os.getenv('EXACT_VLLM_MAX_MODEL_LEN', '4096')
VLLM_GPU_MEMORY_UTILIZATION = os.getenv('EXACT_VLLM_GPU_MEMORY_UTILIZATION', '0.90')
VLLM_WAIT_SECONDS = int(os.getenv('EXACT_VLLM_WAIT_SECONDS', '300'))
VLLM_LOG_PATH = Path(os.getenv('EXACT_VLLM_LOG', str(default_output_dir / 'vllm_server.log')))


def read_log_tail(path: Path, chars: int = 3000) -> str:
    if not path.exists():
        return ''
    return path.read_text(encoding='utf-8', errors='replace')[-chars:]


def wait_for_vllm(base_url: str, timeout_seconds: int, process, log_path: Path) -> None:
    deadline = time.monotonic() + timeout_seconds
    health_url = base_url.rstrip('/') + '/models'
    while time.monotonic() < deadline:
        if process.poll() is not None:
            raise RuntimeError(
                f'vLLM server exited early with code {process.returncode}. Log tail:\n{read_log_tail(log_path)}'
            )
        try:
            with urllib.request.urlopen(health_url, timeout=5) as response:
                if response.status < 500:
                    return
        except (urllib.error.URLError, TimeoutError):
            time.sleep(5)
    raise TimeoutError(
        f'vLLM server did not become ready within {timeout_seconds}s: {health_url}. '
        f'Log tail:\n{read_log_tail(log_path)}'
    )


vllm_proc = None
if START_VLLM:
    if INSTALL_VLLM:
        print('Installing vllm...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'vllm'])

    LLM_BASE_URL = f'http://{VLLM_HOST}:{VLLM_PORT}/v1'
    MODEL_NAME = VLLM_SERVED_MODEL_NAME
    os.environ['EXACT_LLM_BASE_URL'] = LLM_BASE_URL
    os.environ['EXACT_LLM_MODEL'] = MODEL_NAME
    os.environ.setdefault('EXACT_LLM_API_KEY', 'EMPTY')

    command = [
        sys.executable, '-m', 'vllm.entrypoints.openai.api_server',
        '--model', VLLM_MODEL,
        '--served-model-name', VLLM_SERVED_MODEL_NAME,
        '--dtype', VLLM_DTYPE,
        '--max-model-len', VLLM_MAX_MODEL_LEN,
        '--gpu-memory-utilization', VLLM_GPU_MEMORY_UTILIZATION,
        '--enable-prefix-caching',
        '--host', VLLM_HOST,
        '--port', str(VLLM_PORT),
    ]
    if VLLM_QUANTIZATION:
        command.extend(['--quantization', VLLM_QUANTIZATION])

    VLLM_LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
    vllm_log = VLLM_LOG_PATH.open('w', encoding='utf-8')
    vllm_proc = subprocess.Popen(command, stdout=vllm_log, stderr=subprocess.STDOUT, text=True)
    print('started vLLM pid =', vllm_proc.pid)
    print('vLLM log       =', VLLM_LOG_PATH)
    print('vLLM base_url  =', LLM_BASE_URL)
    print('vLLM model     =', MODEL_NAME)
    wait_for_vllm(LLM_BASE_URL, VLLM_WAIT_SECONDS, vllm_proc, VLLM_LOG_PATH)
    print('vLLM ready')
else:
    print('START_VLLM=False; skipping vLLM launch (using Transformers local fallback)')

In [ ]:
def resolve_logic_input() -> Path:
    explicit = os.getenv('EXACT_LOGIC_INPUT', '').strip()
    if explicit:
        path = Path(explicit)
        if path.exists():
            return path
        raise FileNotFoundError(f'EXACT_LOGIC_INPUT does not exist: {path}')

    search_roots = [
        Path('/kaggle/input'),
        PROJECT_ROOT / 'src' / 'exact' / 'datasets' / 'exact',
        PROJECT_ROOT,
    ]
    # For evaluation, prefer the labeled grouped dataset. The inference file may have no gold labels.
    preferred_names = [
        'Logic_Based_Educational_Queries.json',
        'Logic_Based_Educational_Queries_inference.json',
    ]
    for root in search_roots:
        if not root.exists():
            continue
        for name in preferred_names:
            matches = sorted(root.rglob(name)) if root.is_dir() else []
            if matches:
                return matches[0]

    raise FileNotFoundError('Cannot locate Type 1 logic input JSON. Set EXACT_LOGIC_INPUT.')


INPUT_PATH = resolve_logic_input()
print('INPUT_PATH =', INPUT_PATH)

In [ ]:
from exact.common.schemas import PredictionRequest, TaskType, to_official_response
from exact.config import Settings
from exact.datasets.loader import load_logic_dataset
from exact.llm_client import build_json_client_from_settings
from exact.logic.kb import clear_kb_cache
from exact.logic.pipeline import run_type1_pipeline
from exact.router.task_router import TaskRouter
from exact.scripts.evaluate_type1_predictions import evaluate_prediction, summarize, write_errors_csv


def load_type1_examples(path: Path) -> list[dict]:
    payload = json.loads(path.read_text(encoding='utf-8'))

    # Labeled grouped training/dev style: records contain questions/answers arrays.
    records = payload if isinstance(payload, list) else payload.get('data') if isinstance(payload, dict) else None
    if isinstance(records, list) and records and isinstance(records[0], dict) and 'questions' in records[0]:
        df = load_logic_dataset(path)
        examples = []
        for _, row in df.iterrows():
            examples.append({
                'id': row['id'],
                'group_id': row.get('group_id'),
                'question': row['question'],
                'premises-NL': list(row['premises_nl']),
                'gold_answer': str(row.get('gold_answer', '')).strip(),
                'question_type_hint': row.get('question_type'),
            })
        return examples

    # Kaggle inference style: either top-level instances or a list of flat request objects.
    if isinstance(payload, dict) and isinstance(payload.get('instances'), list):
        flat = payload['instances']
    elif isinstance(payload, list):
        flat = payload
    else:
        raise ValueError(f'Unsupported input shape in {path}')

    examples = []
    for index, item in enumerate(flat):
        example = dict(item)
        example.setdefault('id', f'logic_{index:04d}')
        answer = example.get('answer') or example.get('gold_answer') or example.get('label')
        if answer is not None:
            example['gold_answer'] = str(answer).strip()
        examples.append(example)
    return examples


def select_examples(examples: list[dict], limit: int | None, mode: str, seed: int) -> list[dict]:
    if limit is None or limit <= 0 or limit >= len(examples):
        return list(examples)
    rng = random.Random(seed)
    mode = mode.lower()
    if mode == 'head':
        return list(examples[:limit])
    if mode == 'random':
        selected = list(examples)
        rng.shuffle(selected)
        return selected[:limit]
    if mode != 'stratified':
        raise ValueError('EXACT_SAMPLE_MODE must be one of: stratified, random, head')

    buckets: dict[tuple[str, str], list[dict]] = defaultdict(list)
    for example in examples:
        key = (
            str(example.get('question_type_hint') or 'unknown'),
            str(example.get('gold_answer') or 'missing_gold'),
        )
        buckets[key].append(example)
    for bucket in buckets.values():
        rng.shuffle(bucket)

    selected: list[dict] = []
    keys = sorted(buckets)
    while len(selected) < limit and any(buckets[key] for key in keys):
        for key in keys:
            if buckets[key]:
                selected.append(buckets[key].pop())
                if len(selected) >= limit:
                    break
    rng.shuffle(selected)
    return selected


all_examples = load_type1_examples(INPUT_PATH)
examples = select_examples(all_examples, LIMIT, SAMPLE_MODE, SAMPLE_SEED)
if not examples:
    raise ValueError(f'No Type 1 examples loaded from {INPUT_PATH}')

all_df = pd.DataFrame(all_examples)
selected_df = pd.DataFrame(examples)
print('total_examples    =', len(all_examples))
print('selected_examples =', len(examples))
print('full question_type/gold distribution:')
display(pd.crosstab(all_df.get('question_type_hint'), all_df.get('gold_answer'), dropna=False))
print('selected question_type/gold distribution:')
display(pd.crosstab(selected_df.get('question_type_hint'), selected_df.get('gold_answer'), dropna=False))
print('first selected example:')
print(json.dumps({k: examples[0].get(k) for k in ['id', 'question_type_hint', 'question', 'gold_answer']}, ensure_ascii=False, indent=2)[:1200])

In [ ]:
# Build a JSON LLM client through the project codebase.
# EXACT_LLM_BASE_URL uses vLLM/OpenAI-compatible serving; otherwise this falls back to in-process Transformers.
settings = Settings().model_copy(
    update={
        'llm_provider': 'openai' if LLM_BASE_URL else 'local',
        'llm_base_url': LLM_BASE_URL,
        'llm_model': MODEL_NAME,
        'llm_api_key': None,
        'mock_llm': False,
        'llm_max_tokens': MAX_NEW_TOKENS,
        'llm_temperature': LLM_TEMPERATURE,
        'llm_top_p': LLM_TOP_P,
        'llm_device_map': LLM_DEVICE_MAP,
        'llm_torch_dtype': LLM_TORCH_DTYPE,
        'llm_local_files_only': LLM_LOCAL_FILES_ONLY,
        'llm_trust_remote_code': LLM_TRUST_REMOTE_CODE,
        'llm_timeout_seconds': REQUEST_TIMEOUT_SECONDS,
        'type1_translation_samples': TYPE1_TRANSLATION_SAMPLES,
        'type1_sampling_temperature': TYPE1_SAMPLING_TEMPERATURE,
        'type1_enable_cot_fallback': TYPE1_ENABLE_COT_FALLBACK,
    }
)
translator_client = build_json_client_from_settings(settings)
assert translator_client is not None, 'LLM-only evaluation requires a JSON LLM client.'
if CLEAR_KB_CACHE:
    clear_kb_cache()
    print('cleared Type 1 KB cache')
print('translator_client =', type(translator_client).__name__)
print('settings =', settings.model_dump(mode='json', exclude={'llm_api_key'}))

In [ ]:
import concurrent.futures


def normalize_answer(value: object) -> str:
    text = str(value or '').strip()
    return text if CASE_SENSITIVE else text.lower()


def display_answer(value: object) -> str:
    return str(value or '').strip()


def compact_text(value: object, limit: int = 180) -> str:
    text = ' '.join(str(value or '').split())
    return text if len(text) <= limit else text[: limit - 3] + '...'


def should_log_case(index: int, total: int) -> bool:
    return bool(LOG_EVERY) and (index == 1 or index == total or index % LOG_EVERY == 0)


def trace_lines(prediction: dict, prefix: str) -> list[str]:
    return [str(line) for line in prediction.get('cot') or [] if str(line).startswith(prefix)]


def make_error_prediction(example: dict, route_reason: str, error: BaseException) -> dict:
    return {
        'id': example.get('id'),
        'task_type': 'type1_logic',
        'question_type': None,
        'answer': '',
        'explanation': f'Pipeline failed: {error}',
        'fol': None,
        'cot': [],
        'premises': [],
        'confidence': 0.0,
        'error': repr(error),
        'route_reason': route_reason,
        'official': {
            'answer': '',
            'explanation': f'Pipeline failed: {error}',
            'fol': None,
            'cot': [],
            'premises': [],
            'confidence': 0.0,
        },
        'traceback': traceback.format_exc(),
    }


def run_one_example(example: dict) -> tuple[dict, str, object, object]:
    request = PredictionRequest.model_validate(example)
    route = router.route(request)
    if route.task_type != TaskType.TYPE1_LOGIC:
        raise ValueError(f'Expected Type 1 logic request, got {route.task_type}')
    response = run_type1_pipeline(
        request,
        translator_client=translator_client,
        settings=settings,
        question_type=route.question_type,
    )
    prediction = response.model_dump(mode='json')
    prediction['route_reason'] = route.reason
    prediction['official'] = to_official_response(response)
    return prediction, route.reason, request, route


router = TaskRouter()
predictions: list[dict] = []
started_at = time.monotonic()

for index, example in enumerate(examples, start=1):
    route_reason = ''
    request = None
    route = None
    try:
        # Build route once for logging before the timed call.
        request = PredictionRequest.model_validate(example)
        route = router.route(request)
        route_reason = route.reason
        if should_log_case(index, len(examples)):
            print(
                f"[{index}/{len(examples)}] id={request.id} route={route.task_type.value} "
                f"question_type={route.question_type.value} reason={route.reason}"
            )
            print('  question:', compact_text(request.question))
            print('  premises:', len(request.premises_nl or []))

        if USE_THREAD_TIMEOUT:
            with concurrent.futures.ThreadPoolExecutor(max_workers=1) as executor:
                future = executor.submit(run_one_example, example)
                try:
                    prediction, route_reason, request, route = future.result(timeout=REQUEST_TIMEOUT_SECONDS)
                except concurrent.futures.TimeoutError:
                    executor.shutdown(wait=False, cancel_futures=True)
                    raise
        else:
            prediction, route_reason, request, route = run_one_example(example)

        if should_log_case(index, len(examples)):
            print(
                f"  answer={prediction.get('answer')} "
                f"confidence={prediction.get('confidence')} error={bool(prediction.get('error'))}"
            )
            for line in trace_lines(prediction, 'symbolic_consistency_vote:'):
                print(' ', line)
            for line in trace_lines(prediction, 'cot_fallback_after_symbolic_unknown:'):
                print(' ', line)
            if prediction.get('error'):
                print('  pipeline_error:', compact_text(prediction.get('error'), 240))
    except concurrent.futures.TimeoutError as exc:
        if not CONTINUE_ON_ERROR:
            raise TimeoutError(f'Request exceeded {REQUEST_TIMEOUT_SECONDS}s: {example.get("id")}') from exc
        prediction = make_error_prediction(
            example,
            route_reason,
            TimeoutError(f'Request exceeded {REQUEST_TIMEOUT_SECONDS}s'),
        )
        if should_log_case(index, len(examples)):
            print(f"[{index}/{len(examples)}] id={example.get('id')} TIMEOUT after {REQUEST_TIMEOUT_SECONDS}s")
    except Exception as exc:
        if not CONTINUE_ON_ERROR:
            raise
        prediction = make_error_prediction(example, route_reason, exc)
        if should_log_case(index, len(examples)):
            print(f"[{index}/{len(examples)}] id={example.get('id')} FAILED")
            print('  error:', compact_text(repr(exc), 400))
            if LOG_TRACEBACK:
                print(traceback.format_exc())

    if 'gold_answer' in example:
        prediction['gold_answer'] = display_answer(example.get('gold_answer'))
        prediction['is_correct'] = normalize_answer(prediction.get('answer')) == normalize_answer(example.get('gold_answer'))
        if should_log_case(index, len(examples)):
            print(
                f"  gold={prediction['gold_answer']} "
                f"correct={prediction['is_correct']}"
            )

    predictions.append(prediction)
    if PROGRESS_EVERY and (index % PROGRESS_EVERY == 0 or index == len(examples)):
        elapsed = time.monotonic() - started_at
        errors = sum(1 for item in predictions if item.get('error'))
        timeouts = sum('Request exceeded' in str(item.get('error')) for item in predictions if item.get('error'))
        cot_fallbacks = sum(bool(trace_lines(item, 'cot_fallback_after_symbolic_unknown:')) for item in predictions)
        print(
            f'processed {index}/{len(examples)} | errors={errors} | timeouts={timeouts} '
            f'| cot_fallbacks={cot_fallbacks} | elapsed={elapsed:.1f}s'
        )

elapsed = time.monotonic() - started_at
print('done elapsed_seconds =', round(elapsed, 2))

In [ ]:
def build_report(predictions: list[dict]) -> dict:
    symbolic_vote_lines = [
        line
        for item in predictions
        for line in trace_lines(item, 'symbolic_consistency_vote:')
    ]
    cot_fallback_lines = [
        line
        for item in predictions
        for line in trace_lines(item, 'cot_fallback_after_symbolic_unknown:')
    ]
    eval_rows = [evaluate_prediction(item, case_sensitive=CASE_SENSITIVE) for item in predictions]
    eval_summary = summarize(eval_rows)
    report = {
        'source': str(INPUT_PATH),
        'base_url': LLM_BASE_URL,
        'model': MODEL_NAME,
        'max_new_tokens': MAX_NEW_TOKENS,
        'llm_temperature': LLM_TEMPERATURE,
        'llm_top_p': LLM_TOP_P,
        'llm_device_map': LLM_DEVICE_MAP,
        'llm_torch_dtype': LLM_TORCH_DTYPE,
        'llm_local_files_only': LLM_LOCAL_FILES_ONLY,
        'llm_trust_remote_code': LLM_TRUST_REMOTE_CODE,
        'type1_translation_samples': TYPE1_TRANSLATION_SAMPLES,
        'type1_sampling_temperature': TYPE1_SAMPLING_TEMPERATURE,
        'sample_mode': SAMPLE_MODE,
        'sample_seed': SAMPLE_SEED,
        'case_sensitive': CASE_SENSITIVE,
        'count': len(predictions),
        'errors': sum(1 for item in predictions if item.get('error')),
        'cot_fallbacks': len(cot_fallback_lines),
        'symbolic_votes': len(symbolic_vote_lines),
        'answer_counts': dict(Counter(display_answer(item.get('answer')) for item in predictions)),
        'gold_counts': dict(Counter(display_answer(item.get('gold_answer')) for item in predictions if 'gold_answer' in item)),
        'question_type_counts': dict(Counter(str(item.get('question_type')) for item in predictions)),
        'error_kinds': dict(Counter(compact_text(item.get('error'), 120) for item in predictions if item.get('error'))),
        'evaluation': eval_summary,
    }
    # Keep top-level accuracy for quick notebook display.
    report['labeled_count'] = eval_summary['scored_total']
    report['accuracy'] = eval_summary['accuracy']
    report['accuracy_by_question_type'] = eval_summary['by_question_type']
    return report


report = build_report(predictions)
summary_df = pd.json_normalize(report['evaluation'], sep='.')
display(summary_df)
print(json.dumps(report['accuracy_by_question_type'], ensure_ascii=False, indent=2))

In [ ]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
ERRORS_PATH.parent.mkdir(parents=True, exist_ok=True)

output_payload = {
    'source': str(INPUT_PATH),
    'format': 'exact_type1_llm_pipeline_eval',
    'count': len(predictions),
    'predictions': predictions,
}
OUTPUT_PATH.write_text(json.dumps(output_payload, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
REPORT_PATH.write_text(json.dumps(report, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
write_errors_csv(ERRORS_PATH, [evaluate_prediction(item, case_sensitive=CASE_SENSITIVE) for item in predictions])

print('wrote predictions:', OUTPUT_PATH)
print('wrote report     :', REPORT_PATH)
print('wrote errors     :', ERRORS_PATH)
print(json.dumps(report, ensure_ascii=False, indent=2))

In [ ]:
# Inspect wrong/error cases without changing pipeline behavior.
prediction_df = pd.DataFrame(predictions)
failed = prediction_df[prediction_df.get('error').notna()] if 'error' in prediction_df else pd.DataFrame()
wrong = prediction_df[prediction_df.get('is_correct') == False] if 'is_correct' in prediction_df else pd.DataFrame()
print('failed_count =', len(failed))
print('wrong_count  =', len(wrong))

show_cols = [
    col for col in [
        'id', 'question_type', 'answer', 'gold_answer', 'is_correct',
        'confidence', 'error', 'explanation', 'route_reason'
    ]
    if col in prediction_df.columns
]
if len(wrong):
    display(wrong[show_cols].head(20))
elif len(failed):
    display(failed[show_cols].head(20))
else:
    display(prediction_df[show_cols].head(20))